# Regression Discontinuity Designs

Regression Discontinuity Designs (RDDs) are used to estimate the Local Average
Treatment Effect in natural experiments that involve a continuous variable that
*suddenly* show some kind of discontinuous behavior. Examples include students
who are granted a scholarship based on their SAT scores, or policies that get
rolled out at a given timestamp.

The core idea is that observations right around the discontinuity are very
similar to each other, with the only exception being that some observations
are on the left side of the cutoff, while the rest are on the right.

To express this idea mathematically, let $X$ be some continuous random variable
(e.g., time) $X = c$ be the discontinuity (e.g., the exact moment in time a
given policy goes live). Then, the treatment effect at the cutoff, $\tau$, is:

$$\tau = \lim_{x \leftarrow c} E[Y|X=x] - \lim_{x \rightarrow c} E[Y|X=x]$$

---

Imports

In [ ]:
import matplotlib.pyplot as plt
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm

## Key Components

* Running Variable ($X$): A continuous variable used to determine treatment
eligibility.

* Cutoff ($c$): The specific value of the running variable that determines the
discontinuous jump in treatment eligibility.

* The *Centered* Running Variable ($R = X - c$): This is just a transformation
of the original running variable that makes the math cleaner.

* Treatment ($D$): The intervention to be studied.

* Outcome ($Y$): The dependent variable.

## Key Assumptions

* Units cannot choose which side of the discontinuity they're on. In other
words, data points can't choose *when* they get treated (if $X$ is time) or
manipulate their scores (if $X$ represents SAT scores). Mathematically, this
means that observations cannot tamper with their value of $X$.

* There must be no other interventions at the exact same cutoff. Otherwise, we
cannot distinguish which one caused the effect.

* The data points are evenly distributed along the running variable. That is,
there isn't an unusually high density of observations just below or above the
cutoff.


## Sharp RDD

In this design, the treatment status is deterministic.

* $D = 0 if X \lt c$
* $D = 1 if X \ge c$

As we'll see, this is not always the case. For example, there may be students
who have a high enough SAT score to get a scholarship but choose not to take it
(i.e., $x_i \gt c$, but $D_i = 0$). We will talk more about this design later on
in this lecture.

### Basic Form

In its most basic form, the specification is:

$$Y_i = \beta_0 + \beta_1 D_i + \beta_2 R_i + \varepsilon_i$$

We can make this model more flexible by:

* Allowing the *slope* to change after the intervention.
* Allowing for polynomial effects in the running variable (though this will
tend to overfit the data).
* Allowing some kind of non-linear function $f(R)$, such as a spline, kernel
regression, etc.

### Lee (2008)

To illustrate Sharp RDDs, we will use the data from Lee (2008). This article
studies the effect of being the incumbent (i.e., the politician who's currently
holding a given position) on winning the next election.

The core hypothesis is that politicians who win with a very small margin (i.e.,
the share of votes is slightly above the 50% mark) are practically identical
to the runner ups. Hence, the author uses the margin of victory as the running
variable and measures if there's a statistically significant discontinuity
at the 0% mark.

The base model is:

$$Y_{i1} = \beta_0 + \beta_1 Y_{i0} + f(R_i) + \varepsilon$$

Where $t = 0$ represents the current election, $t = 1$ represents the next
election, Y_{it} is a binary variable that shows if candidate $i$ won the
election at time $t$, $f$ is some function of $R$, and $R$ is the margin of
victory measured in percentage points.

---

Read Data

In [ ]:
# Load data
PATH = os.path.join('data', 'lee.csv')
df = pd.read_csv('./data/lee.csv')

# Assign constant
df['const'] = 1

Let's visualize the running variable's distribution (margins of victory). This
bit is important because the whole assumption of quasi-randomness around the
cutoff would be invalidated if units (politicians) could choose or alter their
own value for the running variable.

To test whether the density of observations is continuous at the threshold, we
can run a [McCrary Density Test](
    https://www.sciencedirect.com/science/article/abs/pii/S0304407607001133
). However, judging by the histogram below, there doesn't seem to be an unusual
bunching of observations just above or below the cutoff.

In [ ]:
plt.hist(df['margin'], bins=30)
plt.axvline(x=0, color='C1', ls='--')
plt.title('Running Variable Distribution')
plt.xlabel('Margin (pp)')
plt.ylabel('Frequency')
plt.show()

The response variable is a binary column (win/lose) that we will be modeling
with linear probability models. Plotting the data points as simple 1s and 0s
is not very pleasing.

In [ ]:
plt.scatter(
    x=df['margin'],
    y=df['win_1']
)
plt.axvline(x=0, color='C1', ls='--')
plt.title('Next Election Outcome VS Last Election Margin')
plt.xlabel('Margin (pp)')
plt.ylabel('Next Election Outcome')
plt.show()

Visualizing the data correctly is key in RDDs! The idea is so simple that having
aesthetically pleasing plots is super important for this type of designs.

In the paper, the author bins column `'margin'` and calculates the average
win probability next election. This transformation **IS NOT** used in the
regression models though. It's only used for visualization purposes.

In [ ]:
# Declare the limits of each bin
x_ticks = None

# Declare `bin` column using `pd.qcut`
df['bin'] = None

# Get means per bin
grouped_means = None

# Plot bins
plt.scatter()  # Raw data
plt.scatter()  # Grouped means (centered)
plt.axvline(x=0, color='C1', ls='--')
plt.title('Next Election Outcome VS Last Election Margin')
plt.xlabel('Margin (pp)')
plt.ylabel('Next Election Outcome')
plt.show()

#### Linear Model

Let's run a linear model with a single slope for bot sides.

In [ ]:
# Declare model
m0 = sm.OLS(
    endog=None,
    exog=None,
    hasconst=True
)

# Fit model
r0 = m0.fit(cov_type='HC1')

# View results
print(r0.summary())

Let's run another linear model with heterogeneous margin effects on both sides
of the discontinuity.

In [ ]:
# Declare interaction term
df['margin_win_0'] = None

# Declare model
m1 = sm.OLS(
    endog=None,
    exog=None,
    hasconst=True
)

# Fit model
r1 = m1.fit(cov_type='HC1')

# View results
print(r1.summary())

Let's now run a model with quadratic heterogeneous effects.

In [ ]:
# Declare quadratic terms
df['margin_sq'] = None
df['margin_sq_win_0'] = None

# Declare model
m2 = sm.OLS(
    endog=None,
    exog=None,
    hasconst=True
)

# Fit model
r2 = m2.fit(cov_type='HC1')

# View results
print(r2.summary())